In [1]:
import os
import torch
import numpy as np
import random

def seed_everything(seed):
    """
    Set random seed for reproducibility
    """
    # 1. Python & Numpy
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # 2. PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"🔒 Locked Random Seed: {seed}")


In [2]:



def seed_everything_random():
    """
    Tạo random seed, set seed đó, và return seed để bạn biết
    """
    # Tạo random seed
    random_seed = random.randint(0, 999999)
    
    # Set seed
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    np.random.seed(random_seed)
    random.seed(random_seed)
    
    # Để reproducible
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    return random_seed


In [3]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
#load data
df_men =pd.read_csv(r"/home/ducm/RERUM/dataset/Hillstrom-Men.csv")
df_men = df_men.drop(columns="Unnamed: 0")
print ("---------------------------")
print ("null count:")
print (df_men.isnull().sum())
print ("---------------------------")
print(df_men.dtypes)
print ("---------------------------")
print ("labels:")
print(df_men.columns.tolist())
print ("---------------------------")
print("data shape:")
print(df_men.shape)


---------------------------
null count:
recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
visit              0
conversion         0
spend              0
treatment          0
dtype: int64
---------------------------
recency              int64
history_segment      int64
history            float64
mens                 int64
womens               int64
zip_code            object
newbie               int64
channel             object
visit                int64
conversion           int64
spend              float64
treatment            int64
dtype: object
---------------------------
labels:
['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'visit', 'conversion', 'spend', 'treatment']
---------------------------
data shape:
(42613, 12)


In [5]:
#Hillstrom-men
#split num and cate
cate_cols = ['zip_code', 'channel']
num_cols = ['recency', 'history_segment', 'history']
#split x y t
y_men = df_men["spend"]
t_men = df_men["treatment"]
x_men = df_men.drop(columns=["spend", "treatment", "visit", "conversion"])

# x_men_encode = pd.get_dummies(x_men, columns=cate_cols, drop_first=True)
# x_men_encode = x_men_encode.astype(float)

#train test split - stratify CHỈ bằng treatment
x_men_train, x_men_test, t_men_train, t_men_test, y_men_train, y_men_test = train_test_split(
    x_men, t_men.values, y_men.values,
    test_size=0.3, random_state=42, stratify= t_men
)

# Tạo stratify cho val split - chỉ dùng treatment
stratify_var_train = pd.Series(t_men_train)

x_men_train, x_men_val, t_men_train, t_men_val, y_men_train, y_men_val = train_test_split(
    x_men_train, t_men_train, y_men_train,
    test_size=(1/7), random_state=42, stratify= t_men_train
)

# Fit get_dummies trên train, sau đó align với val/test
x_men_train_encode = pd.get_dummies(x_men_train, columns=cate_cols, drop_first=True)
x_men_val_encode = pd.get_dummies(x_men_val, columns=cate_cols, drop_first=True)
x_men_test_encode = pd.get_dummies(x_men_test, columns=cate_cols, drop_first=True)

# Align columns
x_men_val_encode = x_men_val_encode.reindex(columns=x_men_train_encode.columns, fill_value=0)
x_men_test_encode = x_men_test_encode.reindex(columns=x_men_train_encode.columns, fill_value=0)

scaler = StandardScaler()
x_men_train= scaler.fit_transform(x_men_train_encode)
x_men_val = scaler.transform(x_men_val_encode)
x_men_test = scaler.transform(x_men_test_encode)

print ("✅ Train/Val/Test split with stratification ONLY by treatment (NO DATA LEAKAGE)")
print (f"Train: {x_men_train.shape}, Val: {x_men_val.shape}, Test: {x_men_test.shape}")
print (f"Treatment distribution - Train: {np.mean(t_men_train):.2%}, Val: {np.mean(t_men_val):.2%}, Test: {np.mean(t_men_test):.2%}")
print (f"Spend mean - Train: {np.mean(y_men_train):.2f}, Val: {np.mean(y_men_val):.2f}, Test: {np.mean(y_men_test):.2f}")

# x_men = pd.DataFrame(x_men_train)
x_men_train

✅ Train/Val/Test split with stratification ONLY by treatment (NO DATA LEAKAGE)
Train: (25567, 10), Val: (4262, 10), Test: (12784, 10)
Treatment distribution - Train: 50.00%, Val: 50.00%, Test: 50.00%
Spend mean - Train: 1.00, Val: 0.95, Test: 1.14


array([[-1.36323828, -0.95770591, -0.65534372, ...,  1.22273126,
        -0.87900784,  1.12185957],
       [ 0.63260109, -0.30863193, -0.31367375, ...,  1.22273126,
        -0.87900784,  1.12185957],
       [ 0.34748118, -0.30863193, -0.4344948 , ...,  1.22273126,
         1.13764628, -0.89137716],
       ...,
       [ 0.34748118,  0.34044206,  0.30319097, ..., -0.8178412 ,
        -0.87900784,  1.12185957],
       [-1.07811837,  0.34044206,  0.16319135, ..., -0.8178412 ,
         1.13764628, -0.89137716],
       [-0.79299846, -0.95770591, -0.61911315, ..., -0.8178412 ,
        -0.87900784,  1.12185957]], shape=(25567, 10))

In [6]:
#Transform to tensor
def to_tensor(df):
    return torch.tensor(df, dtype=torch.float32)

x_men_train_t = to_tensor(x_men_train)
x_men_val_t = to_tensor(x_men_val)
x_men_test_t = to_tensor(x_men_test)

y_men_train_t = to_tensor(y_men_train).unsqueeze(1)
y_men_val_t = to_tensor(y_men_val).unsqueeze(1)
y_men_test_t = to_tensor(y_men_test).unsqueeze(1)

t_men_train_t = to_tensor(t_men_train.astype(float)).unsqueeze(1)
t_men_val_t = to_tensor(t_men_val.astype(float)).unsqueeze(1)
t_men_test_t = to_tensor(t_men_test.astype(float)).unsqueeze(1)

# sampler = get_sampler(y_men_train_t, target_positive_ratio=0.2)

#Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 800
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print ("-------------------------------------------------------------")
print ("✅Completed tranform to tensor✅")
print (f"Shape of train: x={x_men_train_t.shape}; y ={y_men_train_t.shape}; t={t_men_train_t.shape}")
print (f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print (f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅Completed tranform to tensor✅
Shape of train: x=torch.Size([25567, 10]); y =torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


Evaluation metrics

In [7]:
from metrics import auuc, auqc, lift, krcc

Build Model

In [8]:
from tarnet import Tarnet

In [9]:
print("📊 Data Distribution Check:")
print(f"Y train: mean={y_men_train.mean():.4f}, std={y_men_train.std():.4f}")
print(f"Y train zeros: {(y_men_train == 0).sum()} / {len(y_men_train)} ({(y_men_train == 0).sum()/len(y_men_train)*100:.1f}%)")
print(f"\nTreatment balance:")
print(f"  Train: {(t_men_train == 1).sum()} treated, {(t_men_train == 0).sum()} control")
print(f"  Test:  {(t_men_test == 1).sum()} treated, {(t_men_test == 0).sum()} control")

📊 Data Distribution Check:
Y train: mean=1.0015, std=14.5993
Y train zeros: 25338 / 25567 (99.1%)

Treatment balance:
  Train: 12784 treated, 12783 control
  Test:  6392 treated, 6392 control


In [10]:
# Import hàm optimize
from searching import optimize_tarnet

# Chạy optimization
study = optimize_tarnet(
    train_loader=train_loader,
    val_loader=val_loader,
    input_dim=x_men_train_t.shape[1],
    n_trials=30,
    verbose=True
)

# Xem kết quả
print(f"Best parameters: {study.best_params}")
print(f"Best Loss: {study.best_value}")

/home/ducm/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-03-06 11:23:28,543] A new study created in memory with name: tarnet_optimization


🚀 OPTUNA HYPERPARAMETER OPTIMIZATION FOR TARNET
📋 Configuration:
   - Number of trials: 30
   - Seeds per trial: 5 [412312, 42, 1874, 902745, 1]
   - Total model trainings: 150
   - Metric to minimize: Loss (on VALIDATION set)
   ⚠️  Using validation set for optimization (no data leakage)

📋 Search Space:
   - uplift_lambda: [0.1, 100.0] (log scale)
   - response_lambda: [1e-4, 10.0] (log scale)
   - lr: [5e-5, 5e-3] (log scale)
   - ranking_start_epoch: [0, 30] (integer)

📋 Fixed Parameters:
   - epochs: 150
   - wd: 1e-4
   - early_stop_metric: loss
   - ema: True, ema_alpha: 0.05
   - patience: 30
   - shared_hidden: 200, outcome_hidden: 100
   - early_stop_start: 50


  0%|          | 0/30 [00:00<?, ?it/s]INFO:searching:
INFO:searching:Trial 1
INFO:searching:  uplift_lambda: 1.329292
INFO:searching:  response_lambda: 5.669850
INFO:searching:  lr: 0.001455
INFO:searching:  ranking_start_epoch: 18
INFO:searching:============================================================
INFO:searching:  Running seed 1/5: 412312


INFO:searching:    Seed 412312 Loss: 375.8273
INFO:searching:  Running seed 2/5: 42
INFO:searching:    Seed 42 Loss: 375.8542
INFO:searching:  Running seed 3/5: 1874
INFO:searching:    Seed 1874 Loss: 375.7598
INFO:searching:  Running seed 4/5: 902745
INFO:searching:    Seed 902745 Loss: 376.0068
INFO:searching:  Running seed 5/5: 1
INFO:searching:    Seed 1 Loss: 375.8771
INFO:searching:
  📊 Trial 1 Results:
INFO:searching:     Mean Loss: 375.8650 ± 0.0811
INFO:searching:     Individual scores: ['375.8273', '375.8542', '375.7598', '376.0068', '375.8771']
Best trial: 0. Best value: 375.865:   3%|▎         | 1/30 [02:31<1:12:59, 151.01s/it]INFO:searching:
INFO:searching:Trial 2
INFO:searching:  uplift_lambda: 0.293803
INFO:searching:  response_lambda: 0.000603
INFO:searching:  lr: 0.000065
INFO:searching:  ranking_start_epoch: 26
INFO:searching:============================================================
INFO:searching:  Running seed 1/5: 412312


[I 2026-03-06 11:25:59,462] Trial 0 finished with value: 375.86503842671715 and parameters: {'uplift_lambda': 1.3292918943162166, 'response_lambda': 5.669849511478847, 'lr': 0.0014553179565665352, 'ranking_start_epoch': 18}. Best is trial 0 with value: 375.86503842671715.


...

[I 2026-03-06 12:41:54,234] Trial 27 finished with value: 356.6105004628499 and parameters: {'uplift_lambda': 48.663280431769756, 'response_lambda': 0.00021205305656079947, 'lr': 0.000449817166707564, 'ranking_start_epoch': 30}. Best is trial 20 with value: 276.9944180806478.


INFO:searching:    Seed 412312 Loss: 376.0432
INFO:searching:  Running seed 2/5: 42
INFO:searching:    Seed 42 Loss: 374.8707
INFO:searching:  Running seed 3/5: 1874
INFO:searching:    Seed 1874 Loss: 375.1107
INFO:searching:  Running seed 4/5: 902745
INFO:searching:    Seed 902745 Loss: 375.9343
INFO:searching:  Running seed 5/5: 1
INFO:searching:    Seed 1 Loss: 372.9529
INFO:searching:
  📊 Trial 29 Results:
INFO:searching:     Mean Loss: 374.9823 ± 1.1117
INFO:searching:     Individual scores: ['376.0432', '374.8707', '375.1107', '375.9343', '372.9529']
Best trial: 20. Best value: 276.994:  97%|█████████▋| 29/30 [1:21:00<02:48, 168.09s/it]INFO:searching:
INFO:searching:Trial 30
INFO:searching:  uplift_lambda: 27.914531
INFO:searching:  response_lambda: 0.000414
INFO:searching:  lr: 0.001221
INFO:searching:  ranking_start_epoch: 17
INFO:searching:============================================================
INFO:searching:  Running seed 1/5: 412312


[I 2026-03-06 12:44:28,542] Trial 28 finished with value: 374.9823345184326 and parameters: {'uplift_lambda': 4.080416607519634, 'response_lambda': 0.001308146688672255, 'lr': 0.0008222361370656895, 'ranking_start_epoch': 19}. Best is trial 20 with value: 276.9944180806478.


INFO:searching:    Seed 412312 Loss: 368.2426
INFO:searching:  Running seed 2/5: 42
INFO:searching:    Seed 42 Loss: 367.1802
INFO:searching:  Running seed 3/5: 1874
INFO:searching:    Seed 1874 Loss: 371.4768
INFO:searching:  Running seed 4/5: 902745
INFO:searching:    Seed 902745 Loss: 338.3109
INFO:searching:  Running seed 5/5: 1
INFO:searching:    Seed 1 Loss: 313.0001
INFO:searching:
  📊 Trial 30 Results:
INFO:searching:     Mean Loss: 351.6421 ± 22.7216
INFO:searching:     Individual scores: ['368.2426', '367.1802', '371.4768', '338.3109', '313.0001']
Best trial: 20. Best value: 276.994: 100%|██████████| 30/30 [1:23:48<00:00, 167.62s/it]

[I 2026-03-06 12:47:17,034] Trial 29 finished with value: 351.64209210475286 and parameters: {'uplift_lambda': 27.914530936094977, 'response_lambda': 0.0004135595090345117, 'lr': 0.0012210308520407276, 'ranking_start_epoch': 17}. Best is trial 20 with value: 276.9944180806478.

🏆 OPTIMIZATION COMPLETE!

📊 Best Trial: #21
   Best Mean Loss (Validation): 276.9944
   ⚠️  Remember to evaluate on TEST set with best params!

🎯 Best Parameters:
   uplift_lambda: 46.463946
   response_lambda: 0.001933
   lr: 0.001922
   ranking_start_epoch: 19

📈 Top 5 Trials:
   Trial 21: Loss=276.9944 | uplift_λ=46.4639 | response_λ=0.001933 | lr=0.001922 | rank_start=19
   Trial 13: Loss=293.7622 | uplift_λ=81.4075 | response_λ=0.006819 | lr=0.000380 | rank_start=0
   Trial 26: Loss=322.4129 | uplift_λ=48.7213 | response_λ=0.000110 | lr=0.000869 | rank_start=24
   Trial 25: Loss=325.3083 | uplift_λ=49.8854 | response_λ=0.000110 | lr=0.000856 | rank_start=23
   Trial 16: Loss=340.9830 | uplift_λ=18.9151 | re